In [ ]:
# This is the code for task 2. As per requirements, we are using sklearn packages here. 
# Also forgot to mention that if you are ever running this locally, ensure that you have installed numpy, pandas, matplotlib as well as sklearn in your venv
# Aim is to use PCA to reduce the dimension of the features.

# From deliverables:
# Need a trained KNN model for its predictions on test after being trained
# Need a test data in PCA space
# Need PCA-reduced train features
# Need train features, labels, test id's as well as X_test
# Need loading of the x, y, X_test, test_ids
# Need shape and id checks
# Need to fit the PCA on train only

In [ ]:
# First starting off with a bit of data analysis to get to know what we are working with here
# Need to check for attributes that would be specific for PCA and therefore would influence the later process
# There are no features that dominate through unit choice, so we dont standardise
# Basic concept here is that PCA is going to try and retain as much variance in the data while it simplifies the dataset as muchas possible

import numpy as np
import pandas as pd
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier

DATA = "50-007-machine-learning-may-2026/"

# This has the same layout as task 1 where train_features.csv has id, label and test_features has id
# Load dataset
train_df = pd.read_csv(DATA + "train_features.csv")
test_df = pd.read_csv(DATA + "test_features.csv")

# .values pulls the raw numpy array out of the df since sklearn needs numbers and not columns. This is being used by the PCA as well as KNN as they are linear algrbra
X = train_df.drop(columns=["id","label"]).values     # These are the train features
y = train_df["label"].values                         # These are the labels
X_test = test_df.drop(columns=["id"]).values         # This is the X_test
test_ids = test_df["id"].values                      # These are the test_ids           

# Check the dataframe
print("X       ", X.shape)
print("X_test  ", X_test.shape)
print("dtype   ", X.dtype)



X        (20000, 5000)
X_test   (6999, 5000)
dtype    float64


In [ ]:
# Fit PCA once at the largest k. Components are sorted and independent of k
# So the first 100 columns of a 2000 component fit are the 100 component fit

pca = PCA(n_components=2000, svd_solver="covariance_eigh", random_state=42)
pca.fit(X)                          # TRAIN ONLY - never X_test

Z_train = pca.transform(X)          # (20000, 2000)
Z_test  = pca.transform(X_test)     # (6999, 2000)

cum = np.cumsum(pca.explained_variance_ratio_)
for k in [100, 500, 1000, 2000]:
    print("k=%5d  cumulative explained variance %.4f" % (k, cum[k-1]))

# From the results, it would seem that if the best 100 of the new features were kept, around 15% of the information is kept
# if best 500 are kept, around 38% of the information is kept
# if best 1000 features of kept, around 56% of the information is kept
# Even if 2000 features are kept, only 78% of the information is kept

k=  100  cumulative explained variance 0.1582
k=  500  cumulative explained variance 0.3895
k= 1000  cumulative explained variance 0.5644
k= 2000  cumulative explained variance 0.7750


In [5]:
# Verification
assert Z_train.shape == (20000, 2000)
assert Z_test.shape  == (6999, 2000)

# The nesting claim
small = PCA(n_components=100, svd_solver="covariance_eigh", random_state=42).fit(X)
print("nested:", np.allclose(np.abs(small.transform(X)), np.abs(Z_train[:, :100])))

nested: True


In [ ]:
# KNN on PCA-reduced features
import time

for k in [2000, 1000, 500, 100]:
    t = time.time()
    knn = KNeighborsClassifier(n_neighbors=2, n_jobs=-1)
    knn.fit(Z_train[:, :k], y)          # KNN "training" just stores the data
    pred = knn.predict(Z_test[:, :k])   # this is where all the time goes

    pd.DataFrame({"id": test_ids, "label": pred}).to_csv(f"pca_{k}_knn.csv", index=False)
    print("k=%5d  %.0fs  predicted class-1 fraction %.4f" % (k, time.time()-t, pred.mean()))

# n_jobs = -1 uses all your cores 

k= 2000  1s  predicted class-1 fraction 0.0582
k= 1000  1s  predicted class-1 fraction 0.1760
k=  500  0s  predicted class-1 fraction 0.3776
k=  100  0s  predicted class-1 fraction 0.5444


In [7]:
check = pd.read_csv("pca_100_knn.csv")
print(check.shape)                      # must be (6999, 2)
print(check.columns.tolist())           # must be ['id', 'label']
print(check["label"].unique())          # must be [0 1] only
print(set(check["id"]) == set(pd.read_csv(DATA + "sample_submission.csv")["id"]))

(6999, 2)
['id', 'label']
[1 0]
True
